In [0]:
# Configura o catálogo e cria o schema da camada Silver

catalog = "workspace"
silver_schema = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

print(f"Schema criado: {catalog}.{silver_schema}")


In [0]:
# Carrega a tabela de informações dos filmes para analisar os dados

df_info = spark.table("workspace.bronze.tb_movies_info")

display(df_info)


In [0]:
# Tratamento da tabela de informações dos filmes

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, coalesce, expr, when,
    row_number, desc_nulls_last, lit, year
)

df_info_silver = (
    df_info
    .withColumn("id", trim(col("id")))
    .withColumn("tconst", trim(col("tconst")))
    .withColumn("title", trim(col("title")))
    .withColumn("original_title", trim(col("original_title")))
    .withColumn("original_language", trim(col("original_language")))
    .withColumn("status", trim(col("status")))
    .withColumn("overview", trim(col("overview")))
    .withColumn("tagline", trim(col("tagline")))

    .withColumn(
        "release_date",
        coalesce(
            expr("try_to_date(release_date, 'yyyy-MM-dd')"),
            expr("try_to_date(release_date, 'MM-dd-yyyy')"),
            expr("try_to_date(release_date, 'dd/MM/yyyy')"),
            expr("try_to_date(release_date, 'MM/dd/yyyy')")
        )
    )


    .withColumn(
        "runtime",
        when(
            trim(col("runtime")).rlike(r"^\d+$"),
            trim(col("runtime")).cast("int")
        )
    )


    .withColumn(
        "status_normalizado",
        trim(
            regexp_replace(
                lower(col("status")),
                r"[\s_-]+",
                " "
            )
        )
    )
    .withColumn(
        "status",
        when(col("status_normalizado") == "released", "Lançado")
        .when(col("status_normalizado") == "post production", "Pós-Produção")
        .when(col("status_normalizado") == "in production", "Em Produção")
        .when(col("status_normalizado") == "planned", "Planejado")
        .when(col("status_normalizado") == "rumored", "Rumores")
        .when(col("status_normalizado") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )
    .drop("status_normalizado")


    .filter(col("id").isNotNull() & (col("id") != ""))
)

campos_informativos = [
    "tconst", "title", "original_title", "original_language",
    "release_date", "runtime", "status", "overview", "tagline"
]

pontuacao = lit(0)

for campo in campos_informativos:
    pontuacao = pontuacao + when(
        col(campo).isNotNull() & (trim(col(campo).cast("string")) != ""),
        1
    ).otherwise(0)

df_info_silver = df_info_silver.withColumn(
    "campos_preenchidos",
    pontuacao
)

janela_filmes = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    col("campos_preenchidos").desc(),
    col("title").asc_nulls_last(),
    col("tconst").asc_nulls_last(),
    col("original_title").asc_nulls_last(),
    col("release_date").asc_nulls_last(),
    col("runtime").asc_nulls_last(),
    col("original_language").asc_nulls_last(),
    col("status").asc_nulls_last(),
    col("overview").asc_nulls_last(),
    col("tagline").asc_nulls_last()
)

df_info_silver = (
    df_info_silver
    .withColumn("rn", row_number().over(janela_filmes))
    .filter(col("rn") == 1)
    .drop("rn", "campos_preenchidos")

    .select(
        col("id").alias("id_filme"),
        col("tconst"),
        col("title").alias("titulo"),
        col("original_title").alias("titulo_original"),
        col("release_date").alias("data_lancamento"),
        year(col("release_date")).alias("ano_lancamento"),
        col("runtime").alias("duracao_minutos"),
        col("original_language").alias("idioma_original"),
        col("status").alias("status_filme"),
        col("overview").alias("sinopse"),
        col("tagline").alias("frase_divulgacao"),
        col("ingestion_datetime")
    )
)




In [0]:
# Carrega os dados financeiros da camada Bronze

df_financeiro = spark.table("workspace.bronze.tb_movies_financials")

print("Quantidade de registros:", df_financeiro.count())

display(df_financeiro.limit(20))


In [0]:
# Carrega os dados de métricas da camada Bronze

df_metricas = spark.table("workspace.bronze.tb_movies_metrics")

print("Quantidade de registros:", df_metricas.count())

display(df_metricas.limit(20))


In [0]:
# Trata as métricas numéricas e avaliações dos filmes

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, when, row_number, desc_nulls_last, lit,
    regexp_replace, instr
)

def converter_numero(coluna):
    valor = trim(col(coluna).cast("string"))

    return (
        when(
            valor.rlike(r"^-?\d+(\.\d+)?$"),
            valor.cast("double")
        )
        .otherwise(None)
    )


def converter_popularidade(coluna):
    valor = regexp_replace(
        trim(col(coluna).cast("string")),
        r"\s+",
        ""
    )

    valor_normalizado = (
        when(
            valor.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
            regexp_replace(valor, ",", "")
        )
        .when(
            valor.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
            regexp_replace(
                regexp_replace(valor, r"\.", ""),
                ",",
                "."
            )
        )
        .when(
            valor.rlike(r"^-?\d+,\d+$"),
            regexp_replace(valor, ",", ".")
        )
        .otherwise(valor)
    )

    return (
        when(
            valor_normalizado.rlike(r"^-?\d+(\.\d+)?$"),
            valor_normalizado.cast("double")
        )
        .otherwise(None)
    )


df_metricas_tratadas = (
    df_metricas
    .withColumn("id", trim(col("id")))
    .withColumn("popularity", converter_popularidade("popularity"))
    .withColumn("vote_average", converter_numero("vote_average"))
    .withColumn("vote_count", converter_numero("vote_count"))
    .withColumn("averageRating", converter_numero("averageRating"))
    .withColumn("numVotes", converter_numero("numVotes"))

    # Descarta registros sem identificador.
    .filter(col("id").isNotNull() & (col("id") != ""))

    # Popularidade não pode ser negativa.
    .withColumn(
        "popularity",
        when(col("popularity") >= 0, col("popularity"))
        .otherwise(None)
    )

    # Notas devem estar entre 0 e 10.
    .withColumn(
        "vote_average",
        when(col("vote_average").between(0, 10), col("vote_average"))
        .otherwise(None)
    )
    .withColumn(
        "averageRating",
        when(col("averageRating").between(0, 10), col("averageRating"))
        .otherwise(None)
    )

    .withColumn(
        "vote_count",
        when(
            (col("vote_count") >= 0) &
            (col("vote_count") == col("vote_count").cast("long")),
            col("vote_count").cast("long")
        ).otherwise(None)
    )
    .withColumn(
        "numVotes",
        when(
            (col("numVotes") >= 0) &
            (col("numVotes") == col("numVotes").cast("long")),
            col("numVotes").cast("long")
        ).otherwise(None)
    )
)

campos_metricas = [
    "popularity", "vote_average", "vote_count",
    "averageRating", "numVotes"
]

pontuacao_metricas = lit(0)

for campo in campos_metricas:
    pontuacao_metricas = (
        pontuacao_metricas
        + when(col(campo).isNotNull(), 1).otherwise(0)
    )

df_metricas_tratadas = df_metricas_tratadas.withColumn(
    "campos_preenchidos",
    pontuacao_metricas
)

janela_metricas = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    col("campos_preenchidos").desc(),
    col("popularity").desc_nulls_last(),
    col("vote_average").desc_nulls_last(),
    col("vote_count").desc_nulls_last(),
    col("averageRating").desc_nulls_last(),
    col("numVotes").desc_nulls_last()
)

df_metricas_silver = (
    df_metricas_tratadas
    .withColumn("rn", row_number().over(janela_metricas))
    .filter(col("rn") == 1)
    .drop("rn", "campos_preenchidos")

    .select(
        col("id").alias("id_filme"),
        col("popularity").alias("popularidade"),
        col("vote_average").alias("nota_media_tmdb"),
        col("vote_count").alias("qtd_votos_tmdb"),
        col("averageRating").alias("nota_media_imdb"),
        col("numVotes").alias("qtd_votos_imdb"),
        col("ingestion_datetime")
    )
)

display(df_metricas_silver.limit(20))


In [0]:
# Carrega os dados de avaliações da camada Bronze

df_avaliacoes = spark.table("workspace.bronze.tb_movies_reviews")

print("Quantidade de registros:", df_avaliacoes.count())

df_avaliacoes.printSchema()

display(df_avaliacoes.limit(20))


In [0]:
# Tratamento dos dados de avaliações para a camada Silver

from pyspark.sql.functions import col, trim, when, expr, lit

df_avaliacoes_silver = (
    df_avaliacoes
    .withColumn("id", trim(col("id")))
    .withColumn("nome", trim(col("nome")))
    .withColumn(
        "comentario",
        when(
            col("comentario").isNull() |
            (trim(col("comentario")) == ""),
            lit("Sem comentário")
        ).otherwise(trim(col("comentario")))
    )
    .withColumn("nota", expr("try_cast(trim(nota) as double)"))
    .withColumn(
        "nota",
        when(col("nota").between(0, 10), col("nota")).otherwise(None)
    )
    .filter(col("id").isNotNull() & (col("id") != ""))
    .select(
        col("id").alias("id_filme"),
        col("nome").alias("nome_usuario"),
        col("nota").alias("nota_usuario"),
        col("comentario").alias("comentario_usuario")
    )
    .dropDuplicates([
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ])
)

display(df_avaliacoes_silver.limit(20))



In [0]:
# Carrega os dados de créditos e tags da camada Bronze

df_credits = spark.table("workspace.bronze.tb_credits_and_tags")

print("Quantidade de registros:", df_credits.count())

df_credits.printSchema()

display(df_credits.limit(20))


In [0]:
# Tratamento dos gêneros dos filmes

from pyspark.sql.functions import col, trim, regexp_replace, split, explode, lower

generos_validos = [
    "action", "adventure", "animation", "comedy", "crime",
    "documentary", "drama", "family", "fantasy", "history",
    "horror", "music", "mystery", "romance", "science fiction",
    "tv movie", "thriller", "war", "western"
]

df_generos_silver = (
    df_credits
    .withColumn("genres", regexp_replace(col("genres"), r"[;|]", ","))
    .withColumn("genero", explode(split(col("genres"), ",")))
    .withColumn("genero", trim(col("genero")))
    .select(
        trim(col("id")).alias("id_filme"),
        col("genero")
    )
    .filter(
        col("id_filme").isNotNull() &
        (col("id_filme") != "") &
        lower(col("genero")).isin(generos_validos)
    )
    .dropDuplicates(["id_filme", "genero"])
)

display(df_generos_silver.limit(30))



In [0]:
# Lista de gêneros válidos encontrados na base
generos_validos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

# Mantém somente os valores que realmente representam gêneros

df_generos_silver = (
    df_generos_silver
    .filter(col("genero").isin(generos_validos))
)

# Confere o resultado final

print("Quantidade de registros:", df_generos_silver.count())
print(
    "Quantidade de gêneros diferentes:",
    df_generos_silver.select("genero").distinct().count()
)

df_generos_silver.select("genero").distinct().orderBy("genero").show(30, truncate=False)


In [0]:
# Carrega os dados de créditos e tags da camada Bronze para iniciar o tratamento de pessoas e empresas

df_pessoas = spark.table("workspace.bronze.tb_credits_and_tags")

df_pessoas.printSchema()

display(df_pessoas.limit(20))


In [0]:
# Extração e tratamento de pessoas e empresas

from pyspark.sql.functions import (
    col, trim, lower, length, lit, explode,
    split, regexp_replace, initcap
)

df_creditos = spark.table("workspace.bronze.tb_credits_and_tags")

def extrair_entidades(df, coluna, tipo):
    return (
        df
        .select(
            trim(col("id")).alias("id_filme"),
            explode(
                split(
                    regexp_replace(col(coluna), r"\s*[|;]\s*", ","),
                    ","
                )
            ).alias("nome_entidade")
        )
        .withColumn("nome_entidade", trim(col("nome_entidade")))
        .filter(
            col("id_filme").isNotNull() &
            (col("id_filme") != "") &
            col("nome_entidade").isNotNull() &
            (col("nome_entidade") != "") &
            (~lower(col("nome_entidade")).isin(
                "n/a", "na", "null", "none", "unknown",
                "não informado", "nao informado", "-"
            )) &
            (length(col("nome_entidade")) <= 100) &
            (~col("nome_entidade").rlike(r"[\\\r\n]")) &
            (~col("nome_entidade").rlike(r"^\d+([.,]\d+)?$"))
        )
        .withColumn("nome_entidade", initcap(col("nome_entidade")))
        .withColumn("tipo_entidade", lit(tipo))
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )

df_empresas = extrair_entidades(
    df_creditos, "production_companies", "Produtora"
)

df_diretores = extrair_entidades(
    df_creditos, "directors", "Diretor"
)

df_roteiristas = extrair_entidades(
    df_creditos, "writers", "Roteirista"
)

df_elenco = extrair_entidades(
    df_creditos, "cast", "Ator"
)

df_pessoas_empresas_silver = (
    df_empresas
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_elenco)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

display(df_pessoas_empresas_silver.limit(30))



In [0]:
# Carrega e verifica a cotação do dólar da camada Bronze
df_cotacao = spark.table("workspace.bronze.tb_cotacao_dolar")

df_cotacao.printSchema()

display(df_cotacao)



In [0]:
# Converte a data/hora da cotação para timestamp e cria a coluna de data que será usada para ordenar e tratar as cotações na camada Silver.

from pyspark.sql.functions import to_timestamp, to_date, col

df_cotacao_tratada = (
    df_cotacao
    .withColumn(
        "dataHoraCotacao",
        to_timestamp(col("dataHoraCotacao"), "yyyy-MM-dd HH:mm:ss.SSSSSS")
    )
    .withColumn(
        "data_cotacao",
        to_date(col("dataHoraCotacao"))
    )
)

display(df_cotacao_tratada.orderBy("data_cotacao"))



In [0]:
# Cria o calendário e preenche os dias sem nova cotação

from pyspark.sql.functions import (
    col, sequence, explode, min, max, last, row_number,
    desc_nulls_last
)
from pyspark.sql.window import Window

# Mantém a cotação mais recente de cada dia

janela_diaria = Window.partitionBy("data_cotacao").orderBy(
    desc_nulls_last("dataHoraCotacao"),
    desc_nulls_last("ingestion_datetime"),
    desc_nulls_last("cotacaoVenda"),
    desc_nulls_last("cotacaoCompra")
)

df_cotacao_diaria = (
    df_cotacao_tratada
    .filter(col("data_cotacao").isNotNull())
    .withColumn("rn", row_number().over(janela_diaria))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Cria uma linha para cada dia do período disponível

df_intervalo = df_cotacao_diaria.select(
    min("data_cotacao").alias("data_inicio"),
    max("data_cotacao").alias("data_fim")
)

df_calendario = df_intervalo.select(
    explode(
        sequence(col("data_inicio"), col("data_fim"))
    ).alias("data_cotacao")
)

df_cotacao_completa = df_calendario.join(
    df_cotacao_diaria,
    on="data_cotacao",
    how="left"
)

# Usa a última cotação conhecida nos dias sem atualização

janela_cotacao = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    df_cotacao_completa
    .withColumn(
        "cotacaoCompra",
        last("cotacaoCompra", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "cotacaoVenda",
        last("cotacaoVenda", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "dataHoraCotacao",
        last("dataHoraCotacao", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "ingestion_datetime",
        last("ingestion_datetime", ignorenulls=True).over(janela_cotacao)
    )
)


In [0]:
# Organiza a estrutura final da cotação para a camada Silver.

df_cotacao_silver = (
    df_cotacao_silver
    .select(
        col("data_cotacao"),
        col("cotacaoCompra").alias("cotacao_compra"),
        col("cotacaoVenda").alias("cotacao_venda"),
        col("dataHoraCotacao").alias("data_hora_cotacao"),
        col("ingestion_datetime")
    )
    .orderBy("data_cotacao")
)

df_cotacao_silver.printSchema()
display(df_cotacao_silver)



In [0]:
# Limpa os valores financeiros e calcula os indicadores em USD e BRL

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, when, regexp_replace, upper, trim, row_number,
    desc_nulls_last, lit
)

def limpar_valor_financeiro(coluna):
    valor = upper(trim(col(coluna).cast("string")))
    numero_limpo = regexp_replace(valor, r"[\$ USD,]", "")

    numero = (
        when(
            valor.isNull() |
            valor.isin("UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", ""),
            lit(None).cast("decimal(18,2)")
        )
        .when(
            valor.rlike(r"^[\$ ]*(USD )?\d+(\.\d+)?M$"),
            (
                regexp_replace(valor, r"[\$ USDM,]", "").cast("decimal(18,2)")
                * lit(1000000)
            ).cast("decimal(18,2)")
        )
        .when(
            valor.rlike(r"^[\$ ]*(USD )?\d+(\.\d+)?K$"),
            (
                regexp_replace(valor, r"[\$ USDK,]", "").cast("decimal(18,2)")
                * lit(1000)
            ).cast("decimal(18,2)")
        )
        .when(
            numero_limpo.rlike(r"^\d+(\.\d+)?$"),
            numero_limpo.cast("decimal(18,2)")
        )
        .otherwise(lit(None).cast("decimal(18,2)"))
    )

    return when(
        numero > 0,
        numero
    ).otherwise(lit(None).cast("decimal(18,2)"))


df_financeiro_tratado = (
    df_financeiro
    .withColumn("id", trim(col("id")))
    .withColumn("budget", limpar_valor_financeiro("budget"))
    .withColumn("revenue", limpar_valor_financeiro("revenue"))
    .filter(col("id").isNotNull() & (col("id") != ""))
)

# Mantém apenas o registro financeiro mais recente de cada filme.
janela_financeiro = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    col("budget").desc_nulls_last(),
    col("revenue").desc_nulls_last()
)

df_financeiro_unico = (
    df_financeiro_tratado
    .withColumn("rn", row_number().over(janela_financeiro))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Utiliza a cotação mais recente disponível como referência para todos os filmes.
df_cotacao_referencia = (
    df_cotacao_silver
    .filter(col("cotacao_venda").isNotNull())
    .orderBy(
        col("data_cotacao").desc(),
        col("data_hora_cotacao").desc()
    )
    .limit(1)
    .select(
        col("cotacao_venda").cast("decimal(18,6)").alias("taxa_cambio")
    )
)

df_financeiro_com_cotacao = (
    df_financeiro_unico
    .withColumnRenamed("id", "id_filme")
    .crossJoin(df_cotacao_referencia)
)

df_financeiro_silver = (
    df_financeiro_com_cotacao
    .withColumn(
        "lucro_usd",
        (col("revenue") - col("budget")).cast("decimal(18,2)")
    )
    .withColumn(
        "margem",
        when(
            col("revenue") > 0,
            (
                col("lucro_usd") / col("revenue") * lit(100)
            ).cast("decimal(10,2)")
        ).otherwise(lit(None).cast("decimal(10,2)"))
    )
    .select(
        col("id_filme"),
        col("budget").alias("orcamento_usd"),
        col("revenue").alias("receita_usd"),
        (col("budget") * col("taxa_cambio"))
            .cast("decimal(18,2)").alias("orcamento_brl"),
        (col("revenue") * col("taxa_cambio"))
            .cast("decimal(18,2)").alias("receita_brl"),
        col("lucro_usd"),
        (col("lucro_usd") * col("taxa_cambio"))
            .cast("decimal(18,2)").alias("lucro_brl"),
        col("margem"),
        col("ingestion_datetime")
    )
)

display(df_financeiro_silver.limit(20))



In [0]:
# Grava os DataFrames tratados como tabelas Delta na camada Silver.

df_info_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_info_filmes")

df_financeiro_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_financeiro_filmes")

df_metricas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_metricas_engajamento")

df_avaliacoes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

df_generos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_generos")

df_pessoas_empresas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_pessoas_empresas")

df_cotacao_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_cotacao_dolar")

print("As 7 tabelas Silver foram gravadas com sucesso.")

